In [3]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
DATA_FILE = ROOT / "data" / "processed" / "mandi_prices_cleaned.parquet"

selection_con = duckdb.connect()

selection_con.execute(f"""
    CREATE VIEW prices AS
    SELECT *
    FROM read_parquet('{DATA_FILE.as_posix()}')
""")

print("Dataset loaded:", DATA_FILE.exists())

dataset_info = selection_con.execute("""
    SELECT
        COUNT(*) AS Total_Records,
        COUNT(DISTINCT Project_Commodity) AS Commodities,
        MIN(Parsed_Date) AS First_Date,
        MAX(Parsed_Date) AS Last_Date
    FROM prices
""").df()

display(dataset_info)

Dataset loaded: True


,Total_Records,Commodities,First_Date,Last_Date
0,9057867,6,2001-01-10,2026-04-20


In [4]:
duplicate_series_dates = selection_con.execute("""
    SELECT
        Project_Commodity,
        State,
        District,
        Market,
        Variety,
        Grade,
        Parsed_Date,
        COUNT(*) AS Records_On_Date
    FROM prices
    WHERE Parsed_Date IS NOT NULL
      AND Modal_Price > 0
    GROUP BY
        Project_Commodity,
        State,
        District,
        Market,
        Variety,
        Grade,
        Parsed_Date
    HAVING COUNT(*) > 1
    ORDER BY Records_On_Date DESC
""").df()

print("Series-date groups with multiple records:",
      len(duplicate_series_dates))

display(duplicate_series_dates.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Series-date groups with multiple records: 0


,Project_Commodity,State,District,Market,Variety,Grade,Parsed_Date,Records_On_Date


In [5]:
series_metrics = selection_con.execute("""
    WITH daily AS (
        SELECT
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,
            Parsed_Date,

            LAG(Parsed_Date) OVER (
                PARTITION BY
                    Project_Commodity,
                    State,
                    District,
                    Market,
                    Variety,
                    Grade
                ORDER BY Parsed_Date
            ) AS Previous_Date

        FROM prices
        WHERE Parsed_Date IS NOT NULL
          AND Modal_Price IS NOT NULL
          AND isfinite(Modal_Price)
          AND Modal_Price > 0
    ),

    series_stats AS (
        SELECT
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,

            COUNT(*) AS Reporting_Days,
            MIN(Parsed_Date) AS First_Date,
            MAX(Parsed_Date) AS Last_Date,

            DATE_DIFF(
                'day',
                MIN(Parsed_Date),
                MAX(Parsed_Date)
            ) + 1 AS Calendar_Span_Days,

            ROUND(
                100.0 * COUNT(*) /
                NULLIF(
                    DATE_DIFF(
                        'day',
                        MIN(Parsed_Date),
                        MAX(Parsed_Date)
                    ) + 1,
                    0
                ),
                2
            ) AS Calendar_Coverage_Pct,

            DATE_DIFF(
                'day',
                MAX(Parsed_Date),
                DATE '2026-04-20'
            ) AS Days_Before_Dataset_End

        FROM daily

        GROUP BY
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade
    ),

    gap_stats AS (
        SELECT
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,

            MAX(
                DATE_DIFF('day', Previous_Date, Parsed_Date)
            ) AS Largest_Gap_Days,

            ROUND(
                MEDIAN(
                    DATE_DIFF('day', Previous_Date, Parsed_Date)
                ),
                2
            ) AS Median_Reporting_Interval,

            ROUND(
                QUANTILE_CONT(
                    DATE_DIFF('day', Previous_Date, Parsed_Date),
                    0.90
                ),
                2
            ) AS P90_Reporting_Interval

        FROM daily
        WHERE Previous_Date IS NOT NULL

        GROUP BY
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade
    )

    SELECT
        s.*,
        g.Largest_Gap_Days,
        g.Median_Reporting_Interval,
        g.P90_Reporting_Interval

    FROM series_stats s

    LEFT JOIN gap_stats g
    USING (
        Project_Commodity,
        State,
        District,
        Market,
        Variety,
        Grade
    )

    ORDER BY
        Project_Commodity,
        Reporting_Days DESC
""").df()

print("Total individual series:", len(series_metrics))

display(series_metrics.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total individual series: 26969


,Project_Commodity,State,District,Market,Variety,Grade,Reporting_Days,First_Date,Last_Date,Calendar_Span_Days,Calendar_Coverage_Pct,Days_Before_Dataset_End,Largest_Gap_Days,Median_Reporting_Interval,P90_Reporting_Interval
0,Cotton,Gujarat,Amreli,Amreli,Other,FAQ,5634,2003-04-05,2025-11-04,8250,68.29,167,87,1.0,2.0
1,Cotton,Karnataka,Bijapur,Bijapur,LH-1556,FAQ,4207,2002-03-06,2025-03-01,8397,50.10,415,364,1.0,3.0
2,Cotton,Gujarat,Bhavnagar,Bhavnagar,Other,FAQ,3984,2003-03-26,2025-10-27,8252,48.28,175,134,1.0,2.0
3,Cotton,Gujarat,Chhota Udaipur,Bodeliu,Shanker 6 (B) 30mm FIne,FAQ,3902,2003-02-14,2025-11-05,8301,47.01,166,464,1.0,2.0
4,Cotton,Karnataka,Raichur,Raichur,F-1054,FAQ,3731,2005-01-07,2025-05-05,7424,50.26,350,361,1.0,3.0
5,Cotton,Gujarat,Mehsana,Visnagar,Other,FAQ,3718,2004-11-18,2025-11-03,7656,48.56,168,190,1.0,3.0
6,Cotton,Rajasthan,Ganganagar,Sriganganagar (Grain),American,FAQ,3639,2002-02-01,2025-11-04,8678,41.93,167,221,1.0,2.0
7,Cotton,Telangana,Khammam,Khammam,Cotton (Unginned),FAQ,3637,2005-11-22,2025-10-28,7281,49.95,174,209,1.0,3.0
8,Cotton,Madhya Pradesh,Badwani,Sendhwa,H4,FAQ,3614,2007-02-19,2023-12-20,6149,58.77,852,134,1.0,2.0
9,Cotton,Tamil Nadu,Madurai,Thirumangalam,MCU 5,FAQ,3598,2005-09-29,2024-06-13,6833,52.66,676,427,1.0,2.0


In [6]:
series_overview = (
    series_metrics
    .groupby("Project_Commodity")
    .agg(
        Total_Series=("Reporting_Days", "size"),
        Median_Reporting_Days=("Reporting_Days", "median"),
        Median_Coverage_Pct=("Calendar_Coverage_Pct", "median"),
        Median_Largest_Gap=("Largest_Gap_Days", "median"),
        Median_Recency_Days=("Days_Before_Dataset_End", "median")
    )
    .round(2)
    .reset_index()
)

display(series_overview)

,Project_Commodity,Total_Series,Median_Reporting_Days,Median_Coverage_Pct,Median_Largest_Gap,Median_Recency_Days
0,Cotton,4767,12.0,22.22,234.0,3788.0
1,Grapes,2560,13.0,16.07,279.0,2207.5
2,Onion,7488,30.0,30.62,70.0,490.5
3,Pomegranate,1762,22.0,21.69,168.5,542.5
4,Soybean,3934,24.0,17.59,98.0,381.0
5,Tomato,6458,36.0,31.83,51.0,326.0


In [7]:
# Candidate selection across different continuity thresholds

base_candidates = series_metrics[
    (series_metrics["Reporting_Days"] >= 365) &
    (series_metrics["Days_Before_Dataset_End"] <= 180)
].copy()

threshold_results = []

for gap_limit in [30, 60, 90, 180]:

    eligible = base_candidates[
        base_candidates["Largest_Gap_Days"] <= gap_limit
    ]

    counts = (
        eligible.groupby("Project_Commodity")
        .size()
        .reindex([
            "Cotton", "Grapes", "Onion",
            "Pomegranate", "Soybean", "Tomato"
        ], fill_value=0)
    )

    for commodity, count in counts.items():
        threshold_results.append({
            "Commodity": commodity,
            "Maximum_Gap_Days": gap_limit,
            "Eligible_Series": count
        })

threshold_comparison = pd.DataFrame(threshold_results)

display(
    threshold_comparison.pivot(
        index="Commodity",
        columns="Maximum_Gap_Days",
        values="Eligible_Series"
    )
)

Maximum_Gap_Days,30,60,90,180
Commodity,,,,
Cotton,1,3,6,11
Grapes,1,4,4,24
Onion,46,96,120,316
Pomegranate,29,35,43,79
Soybean,30,36,39,48
Tomato,80,112,146,295


In [8]:
# Identify candidate series whose market names may refer to Uzhavar Sandhais

market_review = base_candidates.copy()

market_review["Possible_Farmers_Market"] = (
    market_review["Market"]
    .str.contains(
        r"uzhavar|sandhai|santhai",
        case=False,
        na=False
    )
)

# Summary by commodity
market_type_summary = (
    market_review
    .groupby(["Project_Commodity", "Possible_Farmers_Market"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={
        False: "Other_Market_Names",
        True: "Possible_Farmers_Markets"
    })
    .reset_index()
)

display(market_type_summary)

# Inspect the actual market names for manual verification
possible_farmers_markets = (
    market_review[
        market_review["Possible_Farmers_Market"]
    ][
        ["Project_Commodity", "State", "District", "Market",
         "Reporting_Days", "Last_Date", "Largest_Gap_Days"]
    ]
    .drop_duplicates()
    .sort_values(["Project_Commodity", "State", "Market"])
)

display(possible_farmers_markets.to_string(index=False))

Possible_Farmers_Market,Project_Commodity,Other_Market_Names,Possible_Farmers_Markets
0,Cotton,86,0
1,Grapes,29,20
2,Onion,546,147
3,Pomegranate,152,22
4,Soybean,144,0
5,Tomato,490,116


'Project_Commodity      State                  District                               Market  Reporting_Days  Last_Date  Largest_Gap_Days\n           Grapes Tamil Nadu                     Salem           Ammapet (Uzhavar Sandhai )             421 2026-04-14               177\n           Grapes Tamil Nadu                   Madurai           Anaiyur (Uzhavar Sandhai )             448 2026-04-18               161\n           Grapes Tamil Nadu                   Madurai        Anna nagar (Uzhavar Sandhai )             454 2026-04-18               162\n           Grapes Tamil Nadu                   Madurai       Chokkikulam (Uzhavar Sandhai )             445 2026-04-20               156\n           Grapes Tamil Nadu               Krishnagiri             Hosur (Uzhavar Sandhai )             434 2026-04-18               157\n           Grapes Tamil Nadu              Chengalpattu  Jameenrayapettai (Uzhavar Sandhai )             431 2026-04-15               160\n           Grapes Tamil Nadu     

In [9]:
gap_thresholds = [30, 60, 90, 180]

market_type_gap_results = []

for gap_limit in gap_thresholds:

    eligible = market_review[
        market_review["Largest_Gap_Days"] <= gap_limit
    ]

    summary = (
        eligible
        .groupby([
            "Project_Commodity",
            "Possible_Farmers_Market"
        ])
        .size()
        .reset_index(name="Eligible_Series")
    )

    summary["Maximum_Gap_Days"] = gap_limit

    market_type_gap_results.append(summary)

market_type_gap_comparison = pd.concat(
    market_type_gap_results,
    ignore_index=True
)

display(
    market_type_gap_comparison.pivot_table(
        index=[
            "Project_Commodity",
            "Possible_Farmers_Market"
        ],
        columns="Maximum_Gap_Days",
        values="Eligible_Series",
        fill_value=0
    )
)

Maximum_Gap_Days                            30    60     90     180
Project_Commodity Possible_Farmers_Market                          
Cotton            False                     1.0   3.0    6.0   11.0
Grapes            False                     1.0   4.0    4.0    4.0
                  True                      0.0   0.0    0.0   20.0
Onion             False                    43.0  93.0  117.0  169.0
                  True                      3.0   3.0    3.0  147.0
Pomegranate       False                    26.0  32.0   40.0   57.0
                  True                      3.0   3.0    3.0   22.0
Soybean           False                    30.0  36.0   39.0   48.0
Tomato            False                    69.0  96.0  126.0  183.0
                  True                     11.0  16.0   20.0  112.0

In [10]:
gap_distribution = (
    market_review
    .groupby([
        "Project_Commodity",
        "Possible_Farmers_Market"
    ])
    .agg(
        Candidate_Series=("Largest_Gap_Days", "count"),
        Minimum_Gap=("Largest_Gap_Days", "min"),
        Median_Gap=("Largest_Gap_Days", "median"),
        P75_Gap=(
            "Largest_Gap_Days",
            lambda x: x.quantile(0.75)
        ),
        P90_Gap=(
            "Largest_Gap_Days",
            lambda x: x.quantile(0.90)
        ),
        Maximum_Gap=("Largest_Gap_Days", "max")
    )
    .round(1)
    .reset_index()
)

display(gap_distribution)

,Project_Commodity,Possible_Farmers_Market,Candidate_Series,Minimum_Gap,Median_Gap,P75_Gap,P90_Gap,Maximum_Gap
0,Cotton,False,86,10,668.0,1295.2,2159.5,3781
1,Grapes,False,29,6,774.0,1366.0,2411.4,3077
2,Grapes,True,20,156,158.5,160.2,162.4,177
3,Onion,False,546,3,505.5,1299.5,2421.5,6688
4,Onion,True,147,3,159.0,161.0,164.0,173
5,Pomegranate,False,152,5,346.5,908.5,2007.8,4290
6,Pomegranate,True,22,4,159.0,161.0,164.8,167
7,Soybean,False,144,6,662.0,2189.8,4703.7,6615
8,Tomato,False,490,3,345.5,989.8,1915.7,7168
9,Tomato,True,116,5,159.0,163.0,168.5,262


In [11]:
recent_continuity = (
    market_review
    .assign(
        Recent_Reporting_Gap=(
            market_review["Days_Before_Dataset_End"]
            + market_review["Largest_Gap_Days"]
        )
    )
)

# Instead of using the estimated value above as a final metric,
# calculate the actual most recent reporting interval from dates.

recent_continuity_summary = (
    recent_continuity
    .groupby([
        "Project_Commodity",
        "Possible_Farmers_Market"
    ])
    .agg(
        Candidate_Series=("Reporting_Days", "count"),
        Median_Days_Since_Last_Report=(
            "Days_Before_Dataset_End", "median"
        ),
        Median_Historical_Largest_Gap=(
            "Largest_Gap_Days", "median"
        )
    )
    .round(1)
    .reset_index()
)

display(recent_continuity_summary)

,Project_Commodity,Possible_Farmers_Market,Candidate_Series,Median_Days_Since_Last_Report,Median_Historical_Largest_Gap
0,Cotton,False,86,167.0,668.0
1,Grapes,False,29,168.0,774.0
2,Grapes,True,20,1.5,158.5
3,Onion,False,546,167.0,505.5
4,Onion,True,147,1.0,159.0
5,Pomegranate,False,152,168.0,346.5
6,Pomegranate,True,22,1.0,159.0
7,Soybean,False,144,168.0,662.0
8,Tomato,False,490,167.0,345.5
9,Tomato,True,116,2.5,159.0


In [12]:
# Measure reporting continuity during the final 365 days of the dataset

series_keys = [
    "Project_Commodity",
    "State",
    "District",
    "Market",
    "Variety",
    "Grade"
]

# Register the preliminary candidate series
selection_con.register(
    "candidate_keys",
    base_candidates[series_keys].drop_duplicates()
)

recent_interval_stats = selection_con.execute("""
    WITH candidate_daily AS (
        SELECT DISTINCT
            p.Project_Commodity,
            p.State,
            p.District,
            p.Market,
            p.Variety,
            p.Grade,
            p.Parsed_Date
        FROM prices p
        INNER JOIN candidate_keys c
        USING (
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade
        )
        WHERE p.Parsed_Date IS NOT NULL
          AND p.Modal_Price IS NOT NULL
          AND isfinite(p.Modal_Price)
          AND p.Modal_Price > 0
    ),

    intervals AS (
        SELECT *,
            LAG(Parsed_Date) OVER (
                PARTITION BY
                    Project_Commodity,
                    State,
                    District,
                    Market,
                    Variety,
                    Grade
                ORDER BY Parsed_Date
            ) AS Previous_Date
        FROM candidate_daily
    ),

    recent_stats AS (
        SELECT
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,

            COUNT(*) FILTER (
                WHERE Parsed_Date >= DATE '2025-04-21'
            ) AS Recent_Reporting_Days,

            MAX(
                DATE_DIFF('day', Previous_Date, Parsed_Date)
            ) FILTER (
                WHERE Parsed_Date >= DATE '2025-04-21'
                  AND Previous_Date IS NOT NULL
                  AND Previous_Date >= DATE '2025-04-20'
            ) AS Recent_Largest_Interval

        FROM intervals
        GROUP BY
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade
    )

    SELECT *
    FROM recent_stats
""").df()

print("Candidate series analysed:", len(recent_interval_stats))
display(recent_interval_stats.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Candidate series analysed: 1752


,Project_Commodity,State,District,Market,Variety,Grade,Recent_Reporting_Days,Recent_Largest_Interval
0,Onion,Haryana,Karnal,Gharaunda,Other,FAQ,145,7
1,Onion,Haryana,Rohtak,Meham,Other,FAQ,142,7
2,Onion,Haryana,Yamuna Nagar,Yamuna Nagar,Other,FAQ,73,63
3,Onion,Punjab,Bhatinda,Raman,Other,FAQ,152,7
4,Onion,Punjab,Mansa,Budalada,Other,FAQ,82,15


In [15]:
# Combine recent reporting metrics with market classification

recent_review = recent_interval_stats.merge(
    market_review[
        series_keys + ["Possible_Farmers_Market"]
    ],
    on=series_keys,
    how="left"
)

# Summarise recent reporting behaviour by commodity and market type

recent_continuity_summary = (
    recent_review
    .groupby([
        "Project_Commodity",
        "Possible_Farmers_Market"
    ])
    .agg(
        Candidate_Series=("Recent_Reporting_Days", "count"),

        Median_Reporting_Days=(
            "Recent_Reporting_Days", "median"
        ),

        P25_Reporting_Days=(
            "Recent_Reporting_Days",
            lambda x: x.quantile(0.25)
        ),

        P75_Reporting_Days=(
            "Recent_Reporting_Days",
            lambda x: x.quantile(0.75)
        ),

        Median_Largest_Interval=(
            "Recent_Largest_Interval", "median"
        ),

        P90_Largest_Interval=(
            "Recent_Largest_Interval",
            lambda x: x.quantile(0.90)
        ),

        Series_With_Gap_Over_30_Days=(
            "Recent_Largest_Interval",
            lambda x: (x > 30).sum()
        ),

        Series_With_Gap_Over_60_Days=(
            "Recent_Largest_Interval",
            lambda x: (x > 60).sum()
        )
    )
    .round(1)
    .reset_index()
)

display(recent_continuity_summary)

,Project_Commodity,Possible_Farmers_Market,Candidate_Series,Median_Reporting_Days,P25_Reporting_Days,P75_Reporting_Days,Median_Largest_Interval,P90_Largest_Interval,Series_With_Gap_Over_30_Days,Series_With_Gap_Over_60_Days
0,Cotton,False,86,32.5,13.0,92.2,124.5,233.5,62,54
1,Grapes,False,29,65.0,37.0,102.0,35.0,110.8,15,11
2,Grapes,True,20,174.5,170.0,178.0,158.5,162.4,20,20
3,Onion,False,546,138.0,111.2,161.0,9.0,26.5,50,31
4,Onion,True,147,166.0,164.0,169.0,159.0,164.0,144,144
5,Pomegranate,False,152,127.0,93.0,147.2,9.0,46.9,25,11
6,Pomegranate,True,22,167.5,164.5,173.0,159.0,164.8,19,19
7,Soybean,False,144,109.0,67.0,127.0,12.0,92.4,28,19
8,Tomato,False,490,142.0,115.0,162.0,9.0,47.0,70,42
9,Tomato,True,116,159.0,116.0,175.0,159.0,168.5,105,100


In [14]:
recent_interval_stats = selection_con.execute("""
    WITH candidate_daily AS (
        SELECT DISTINCT
            p.Project_Commodity,
            p.State,
            p.District,
            p.Market,
            p.Variety,
            p.Grade,
            p.Parsed_Date
        FROM prices p
        INNER JOIN candidate_keys c
        USING (
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade
        )
        WHERE p.Parsed_Date IS NOT NULL
          AND p.Modal_Price IS NOT NULL
          AND isfinite(p.Modal_Price)
          AND p.Modal_Price > 0
    ),

    intervals AS (
        SELECT *,
            LAG(Parsed_Date) OVER (
                PARTITION BY
                    Project_Commodity,
                    State,
                    District,
                    Market,
                    Variety,
                    Grade
                ORDER BY Parsed_Date
            ) AS Previous_Date
        FROM candidate_daily
    )

    SELECT
        Project_Commodity,
        State,
        District,
        Market,
        Variety,
        Grade,

        COUNT(*) FILTER (
            WHERE Parsed_Date >= DATE '2025-04-21'
              AND Parsed_Date <= DATE '2026-04-20'
        ) AS Recent_Reporting_Days,

        MAX(
            DATE_DIFF('day', Previous_Date, Parsed_Date)
        ) FILTER (
            WHERE Parsed_Date >= DATE '2025-04-21'
              AND Parsed_Date <= DATE '2026-04-20'
              AND Previous_Date IS NOT NULL
        ) AS Recent_Largest_Interval

    FROM intervals
    GROUP BY
        Project_Commodity,
        State,
        District,
        Market,
        Variety,
        Grade
""").df()

print("Candidate series analysed:", len(recent_interval_stats))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Candidate series analysed: 1752


In [17]:
# Merge corrected recent metrics with historical candidate metrics
# and add market classification

candidate_review = (
    base_candidates
    .merge(
        recent_interval_stats,
        on=series_keys,
        how="left"
    )
    .merge(
        market_review[
            series_keys + ["Possible_Farmers_Market"]
        ],
        on=series_keys,
        how="left"
    )
)

# Compare candidate counts under different recent reporting requirements

reporting_thresholds = [30, 60, 90, 120]
gap_thresholds = [30, 60, 90]

selection_results = []

for commodity in [
    "Cotton", "Grapes", "Onion",
    "Pomegranate", "Soybean", "Tomato"
]:
    commodity_data = candidate_review[
        (candidate_review["Project_Commodity"] == commodity)
        & (~candidate_review["Possible_Farmers_Market"])
    ]

    for min_days in reporting_thresholds:
        for max_gap in gap_thresholds:

            eligible = commodity_data[
                (commodity_data["Recent_Reporting_Days"] >= min_days)
                & (
                    commodity_data["Recent_Largest_Interval"]
                    <= max_gap
                )
            ]

            selection_results.append({
                "Commodity": commodity,
                "Minimum_Recent_Days": min_days,
                "Maximum_Recent_Gap": max_gap,
                "Eligible_Series": len(eligible)
            })

selection_comparison = pd.DataFrame(selection_results)

display(
    selection_comparison.pivot_table(
        index=["Commodity", "Minimum_Recent_Days"],
        columns="Maximum_Recent_Gap",
        values="Eligible_Series",
        fill_value=0
    )
)

Maximum_Recent_Gap                  30     60     90
Commodity   Minimum_Recent_Days                     
Cotton      30                    24.0   32.0   37.0
            60                    23.0   27.0   29.0
            90                    22.0   22.0   22.0
            120                   19.0   19.0   19.0
Grapes      30                    14.0   17.0   19.0
            60                    13.0   14.0   15.0
            90                     8.0    8.0    8.0
            120                    5.0    5.0    5.0
Onion       30                   494.0  511.0  520.0
            60                   474.0  483.0  489.0
            90                   448.0  454.0  457.0
            120                  376.0  378.0  378.0
Pomegranate 30                   126.0  136.0  138.0
            60                   124.0  130.0  132.0
            90                   115.0  116.0  116.0
            120                   86.0   86.0   86.0
Soybean     30                   116.0  123.0  123.0
            60                   110.0  111.0  111.0
            90                    91.0   91.0   91.0
            120                   62.0   62.0   62.0
Tomato      30                   419.0  441.0  448.0
            60                   413.0  430.0  436.0
            90                   388.0  396.0  398.0
            120                  347.0  350.0  350.0

In [18]:
# Examine the effect of requiring more recent market activity

recency_thresholds = [30, 60, 90, 180]

recency_results = []

for commodity in [
    "Cotton", "Grapes", "Onion",
    "Pomegranate", "Soybean", "Tomato"
]:
    commodity_data = candidate_review[
        (candidate_review["Project_Commodity"] == commodity)
        & (~candidate_review["Possible_Farmers_Market"])
        & (candidate_review["Recent_Reporting_Days"] >= 60)
        & (candidate_review["Recent_Largest_Interval"] <= 60)
    ]

    for max_days_since_report in recency_thresholds:

        eligible = commodity_data[
            commodity_data["Days_Before_Dataset_End"]
            <= max_days_since_report
        ]

        recency_results.append({
            "Commodity": commodity,
            "Maximum_Days_Since_Last_Report": max_days_since_report,
            "Eligible_Series": len(eligible)
        })

recency_comparison = pd.DataFrame(recency_results)

display(
    recency_comparison.pivot(
        index="Commodity",
        columns="Maximum_Days_Since_Last_Report",
        values="Eligible_Series"
    )
)

Maximum_Days_Since_Last_Report,30,60,90,180
Commodity,,,,
Cotton,0,0,0,27
Grapes,0,0,0,14
Onion,0,0,0,483
Pomegranate,0,0,0,130
Soybean,0,0,0,111
Tomato,0,0,0,430


In [19]:
# Inspect last-report dates for candidates passing recent quality criteria

eligible_60_60 = candidate_review[
    (~candidate_review["Possible_Farmers_Market"])
    & (candidate_review["Recent_Reporting_Days"] >= 60)
    & (candidate_review["Recent_Largest_Interval"] <= 60)
].copy()

last_report_summary = (
    eligible_60_60
    .groupby("Project_Commodity")
    .agg(
        Eligible_Series=("Last_Date", "count"),
        Earliest_Last_Report=("Last_Date", "min"),
        Median_Last_Report=("Last_Date", "median"),
        Latest_Last_Report=("Last_Date", "max"),
        Median_Days_Since_Report=(
            "Days_Before_Dataset_End", "median"
        ),
        Latest_Days_Since_Report=(
            "Days_Before_Dataset_End", "min"
        )
    )
    .reset_index()
)

display(last_report_summary)

,Project_Commodity,Eligible_Series,Earliest_Last_Report,Median_Last_Report,Latest_Last_Report,Median_Days_Since_Report,Latest_Days_Since_Report
0,Cotton,27,2025-10-27,2025-11-05 00:00:00,2025-11-05,166.0,166
1,Grapes,14,2025-10-22,2025-11-02 12:00:00,2025-11-05,168.5,166
2,Onion,483,2025-10-22,2025-11-04 00:00:00,2025-11-06,167.0,165
3,Pomegranate,130,2025-10-22,2025-11-03 00:00:00,2025-11-06,168.0,165
4,Soybean,111,2025-10-23,2025-11-03 00:00:00,2025-11-05,168.0,166
5,Tomato,430,2025-10-22,2025-11-04 00:00:00,2025-11-06,167.0,165


In [20]:
# Check reporting continuation after 6 November 2025
# among all series with at least 365 historical reporting days.

established_series = series_metrics[
    series_metrics["Reporting_Days"] >= 365
].copy()

continuation_summary = (
    established_series
    .groupby("Project_Commodity")
    .agg(
        Established_Series=("Reporting_Days", "size"),

        Series_Reported_After_Nov_06=(
            "Last_Date",
            lambda x: (x > pd.Timestamp("2025-11-06")).sum()
        ),

        Series_Reported_After_Dec_31=(
            "Last_Date",
            lambda x: (x > pd.Timestamp("2025-12-31")).sum()
        ),

        Series_Reported_After_Mar_31=(
            "Last_Date",
            lambda x: (x > pd.Timestamp("2026-03-31")).sum()
        ),

        Latest_Report=("Last_Date", "max")
    )
    .reset_index()
)

display(continuation_summary)

,Project_Commodity,Established_Series,Series_Reported_After_Nov_06,Series_Reported_After_Dec_31,Series_Reported_After_Mar_31,Latest_Report
0,Cotton,659,0,0,0,2025-11-05
1,Grapes,292,20,20,20,2026-04-20
2,Onion,1794,144,144,144,2026-04-20
3,Pomegranate,380,19,19,19,2026-04-20
4,Soybean,827,0,0,0,2025-11-05
5,Tomato,1609,97,97,97,2026-04-20


In [21]:
# Identify established series that continued reporting after 6 November 2025

post_nov_series = series_metrics[
    (series_metrics["Reporting_Days"] >= 365)
    & (series_metrics["Last_Date"] > pd.Timestamp("2025-11-06"))
].copy()

post_nov_series["Possible_Farmers_Market"] = (
    post_nov_series["Market"]
    .str.contains(
        r"uzhavar|sandhai|santhai",
        case=False,
        na=False
    )
)

post_nov_summary = (
    post_nov_series
    .groupby(["Project_Commodity", "Possible_Farmers_Market"])
    .agg(
        Series_Count=("Reporting_Days", "size"),
        Median_Reporting_Days=("Reporting_Days", "median"),
        Median_Last_Report=("Last_Date", "median"),
        Latest_Report=("Last_Date", "max")
    )
    .reset_index()
)

display(post_nov_summary)

,Project_Commodity,Possible_Farmers_Market,Series_Count,Median_Reporting_Days,Median_Last_Report,Latest_Report
0,Grapes,True,20,443.0,2026-04-18 12:00:00,2026-04-20
1,Onion,True,144,441.0,2026-04-19 00:00:00,2026-04-20
2,Pomegranate,True,19,441.0,2026-04-19 00:00:00,2026-04-20
3,Tomato,False,1,465.0,2026-04-17 00:00:00,2026-04-17
4,Tomato,True,96,442.0,2026-04-19 00:00:00,2026-04-20


In [22]:
# Inspect the surviving non-Uzhavar market and sample Uzhavar markets

print("Surviving non-Uzhavar series:")

display(
    post_nov_series[
        ~post_nov_series["Possible_Farmers_Market"]
    ][
        [
            "Project_Commodity",
            "State",
            "District",
            "Market",
            "Variety",
            "Grade",
            "Reporting_Days",
            "First_Date",
            "Last_Date",
            "Largest_Gap_Days"
        ]
    ].to_string(index=False)
)

print("\nPossible Uzhavar Sandhai market distribution:")

display(
    post_nov_series[
        post_nov_series["Possible_Farmers_Market"]
    ]
    .groupby(["State", "Market"])
    .size()
    .reset_index(name="Series_Count")
    .sort_values("Series_Count", ascending=False)
    .head(20)
    .to_string(index=False)
)

Surviving non-Uzhavar series:


'Project_Commodity   State      District     Market Variety Grade  Reporting_Days First_Date  Last_Date  Largest_Gap_Days\n           Tomato Tripura North Tripura Kanchanpur  Hybrid   FAQ             465 2020-04-29 2026-04-17               844'


Possible Uzhavar Sandhai market distribution:


'     State                              Market  Series_Count\nTamil Nadu          Anaiyur (Uzhavar Sandhai )             4\nTamil Nadu       Anna nagar (Uzhavar Sandhai )             4\nTamil Nadu     Mannargudi I (Uzhavar Sandhai )             4\nTamil Nadu            Hosur (Uzhavar Sandhai )             4\nTamil Nadu      Chokkikulam (Uzhavar Sandhai )             4\nTamil Nadu   Udhagamandalam (Uzhavar Sandhai )             4\nTamil Nadu        Tuticorin (Uzhavar Sandhai )             4\nTamil Nadu        Thanjavur (Uzhavar Sandhai )             4\nTamil Nadu            Theni (Uzhavar Sandhai )             3\nTamil Nadu   Tiruvannamalai (Uzhavar Sandhai )             3\nTamil Nadu    Palayamkottai (Uzhavar Sandhai )             3\nTamil Nadu          RSPuram (Uzhavar Sandhai )             3\nTamil Nadu        Vadavalli (Uzhavar Sandhai )             3\nTamil Nadu Jameenrayapettai (Uzhavar Sandhai )             3\nTamil Nadu          Katpadi (Uzhavar Sandhai )             3\nTamil N

In [23]:
# Inspect the only established non-Uzhavar series
# that reported after 6 November 2025.

display(
    post_nov_series[
        ~post_nov_series["Possible_Farmers_Market"]
    ][
        [
            "Project_Commodity",
            "State",
            "District",
            "Market",
            "Variety",
            "Grade",
            "Reporting_Days",
            "First_Date",
            "Last_Date",
            "Days_Before_Dataset_End",
            "Largest_Gap_Days",
            "Median_Reporting_Interval",
            "P90_Reporting_Interval"
        ]
    ].to_string(index=False)
)

'Project_Commodity   State      District     Market Variety Grade  Reporting_Days First_Date  Last_Date  Days_Before_Dataset_End  Largest_Gap_Days  Median_Reporting_Interval  P90_Reporting_Interval\n           Tomato Tripura North Tripura Kanchanpur  Hybrid   FAQ             465 2020-04-29 2026-04-17                        3               844                        1.0                     2.0'

In [24]:
# Inspect reporting gaps in the Kanchanpur Tomato series

kanchanpur_gaps = selection_con.execute("""
    WITH dates AS (
        SELECT DISTINCT Parsed_Date
        FROM prices
        WHERE Project_Commodity = 'Tomato'
          AND State = 'Tripura'
          AND District = 'North Tripura'
          AND Market = 'Kanchanpur'
          AND Variety = 'Hybrid'
          AND Grade = 'FAQ'
          AND Parsed_Date IS NOT NULL
          AND Modal_Price > 0
          AND isfinite(Modal_Price)
    ),
    intervals AS (
        SELECT
            Parsed_Date,
            LAG(Parsed_Date) OVER (
                ORDER BY Parsed_Date
            ) AS Previous_Date
        FROM dates
    )
    SELECT
        Previous_Date,
        Parsed_Date AS Next_Report_Date,
        DATE_DIFF('day', Previous_Date, Parsed_Date)
            AS Gap_Interval_Days
    FROM intervals
    WHERE Previous_Date IS NOT NULL
    ORDER BY Gap_Interval_Days DESC
    LIMIT 10
""").df()

display(kanchanpur_gaps)

,Previous_Date,Next_Report_Date,Gap_Interval_Days
0,2020-10-05,2023-01-27,844
1,2023-07-16,2024-05-10,299
2,2025-04-17,2025-12-04,231
3,2020-06-06,2020-09-06,92
4,2026-02-01,2026-04-06,64
5,2023-02-06,2023-03-18,40
6,2020-09-15,2020-10-01,16
7,2025-03-26,2025-04-09,14
8,2025-12-17,2025-12-27,10
9,2026-01-17,2026-01-25,8


In [25]:
# Evaluate historical gap severity among established conventional markets

conventional_series = series_metrics[
    (series_metrics["Reporting_Days"] >= 365)
    & (series_metrics["Last_Date"] > pd.Timestamp("2025-11-06"))
].copy()

conventional_series = conventional_series[
    ~conventional_series["Market"].str.contains(
        r"uzhavar|sandhai|santhai",
        case=False,
        na=False
    )
]

gap_thresholds = [30, 60, 90, 180, 365]

gap_results = []

for commodity in [
    "Cotton", "Grapes", "Onion",
    "Pomegranate", "Soybean", "Tomato"
]:

    commodity_data = conventional_series[
        conventional_series["Project_Commodity"] == commodity
    ]

    for threshold in gap_thresholds:

        gap_results.append({
            "Commodity": commodity,
            "Maximum_Historical_Gap": threshold,
            "Eligible_Series": (
                commodity_data["Largest_Gap_Days"] <= threshold
            ).sum()
        })

gap_comparison = pd.DataFrame(gap_results)

display(
    gap_comparison.pivot(
        index="Commodity",
        columns="Maximum_Historical_Gap",
        values="Eligible_Series"
    )
)

Maximum_Historical_Gap,30,60,90,180,365
Commodity,,,,,
Cotton,0,0,0,0,0
Grapes,0,0,0,0,0
Onion,0,0,0,0,0
Pomegranate,0,0,0,0,0
Soybean,0,0,0,0,0
Tomato,0,0,0,0,0


In [26]:
# Evaluate historical gap severity across ALL established
# conventional market series, regardless of last reporting date.

all_conventional_series = series_metrics[
    series_metrics["Reporting_Days"] >= 365
].copy()

all_conventional_series = all_conventional_series[
    ~all_conventional_series["Market"].str.contains(
        r"uzhavar|sandhai|santhai",
        case=False,
        na=False
    )
].copy()

gap_thresholds = [30, 60, 90, 180, 365, 730]

gap_results = []

for commodity in [
    "Cotton", "Grapes", "Onion",
    "Pomegranate", "Soybean", "Tomato"
]:

    commodity_data = all_conventional_series[
        all_conventional_series["Project_Commodity"] == commodity
    ]

    for threshold in gap_thresholds:

        gap_results.append({
            "Commodity": commodity,
            "Maximum_Historical_Gap": threshold,
            "Eligible_Series": (
                commodity_data["Largest_Gap_Days"] <= threshold
            ).sum()
        })

gap_comparison = pd.DataFrame(gap_results)

display(
    gap_comparison.pivot(
        index="Commodity",
        columns="Maximum_Historical_Gap",
        values="Eligible_Series"
    )
)

Maximum_Historical_Gap,30,60,90,180,365,730
Commodity,,,,,,
Cotton,16,38,76,114,260,397
Grapes,5,8,9,22,88,143
Onion,105,252,325,489,736,1055
Pomegranate,42,57,76,118,173,237
Soybean,191,216,224,280,382,504
Tomato,151,240,323,474,707,977


In [27]:
# Combine historical and recent reporting criteria

combined_candidates = candidate_review[
    (~candidate_review["Possible_Farmers_Market"])
    & (candidate_review["Reporting_Days"] >= 365)
    & (candidate_review["Largest_Gap_Days"] <= 90)
    & (candidate_review["Recent_Reporting_Days"] >= 60)
    & (candidate_review["Recent_Largest_Interval"] <= 60)
].copy()

combined_summary = (
    combined_candidates
    .groupby("Project_Commodity")
    .agg(
        Eligible_Series=("Reporting_Days", "size"),
        Median_Reporting_Days=("Reporting_Days", "median"),
        Median_Historical_Gap=("Largest_Gap_Days", "median"),
        Median_Recent_Days=("Recent_Reporting_Days", "median"),
        Latest_Report=("Last_Date", "max")
    )
    .reindex([
        "Cotton", "Grapes", "Onion",
        "Pomegranate", "Soybean", "Tomato"
    ])
    .reset_index()
)

display(combined_summary)

,Project_Commodity,Eligible_Series,Median_Reporting_Days,Median_Historical_Gap,Median_Recent_Days,Latest_Report
0,Cotton,6,2169.0,55.5,127.5,2025-11-05
1,Grapes,4,1116.0,39.0,145.5,2025-11-05
2,Onion,112,1828.0,33.0,150.0,2025-11-05
3,Pomegranate,39,1727.0,14.0,144.0,2025-11-05
4,Soybean,38,469.5,14.0,125.5,2025-11-05
5,Tomato,121,1652.0,27.0,152.0,2025-11-05


In [28]:
# Compare monthly reporting activity by market type
# from October 2025 through April 2026.

monthly_market_type = selection_con.execute("""
    SELECT
        DATE_TRUNC('month', Parsed_Date) AS Report_Month,

        CASE
            WHEN REGEXP_MATCHES(
                Market,
                '(?i)uzhavar|sandhai|santhai'
            )
            THEN 'Possible Uzhavar Sandhai'
            ELSE 'Other Market Names'
        END AS Market_Type,

        Project_Commodity,

        COUNT(*) AS Records,

        COUNT(DISTINCT State || '|' || District || '|' || Market)
            AS Active_Markets

    FROM prices

    WHERE Parsed_Date >= DATE '2025-10-01'
      AND Parsed_Date <= DATE '2026-04-20'

    GROUP BY
        Report_Month,
        Market_Type,
        Project_Commodity

    ORDER BY
        Report_Month,
        Market_Type,
        Project_Commodity
""").df()

display(monthly_market_type)

,Report_Month,Market_Type,Project_Commodity,Records,Active_Markets
0,2025-10-01,Other Market Names,Cotton,2296,188
1,2025-10-01,Other Market Names,Grapes,443,47
2,2025-10-01,Other Market Names,Onion,11346,811
3,2025-10-01,Other Market Names,Pomegranate,2780,246
4,2025-10-01,Other Market Names,Soybean,4802,452
...,...,...,...,...,...
71,2026-04-01,Possible Uzhavar Sandhai,Grapes,210,57
72,2026-04-01,Possible Uzhavar Sandhai,Onion,1308,323
73,2026-04-01,Possible Uzhavar Sandhai,Pomegranate,210,66
74,2026-04-01,Possible Uzhavar Sandhai,Soybean,79,24


In [29]:
# Summarise monthly reporting volume by market type

monthly_type_summary = (
    monthly_market_type
    .groupby(["Report_Month", "Market_Type"])
    .agg(
        Total_Records=("Records", "sum"),
        Market_Commodity_Count=("Active_Markets", "sum")
    )
    .reset_index()
)

display(
    monthly_type_summary.pivot(
        index="Report_Month",
        columns="Market_Type",
        values="Total_Records"
    ).fillna(0).astype(int)
)

print("\nActive market-commodity counts:")

display(
    monthly_type_summary.pivot(
        index="Report_Month",
        columns="Market_Type",
        values="Market_Commodity_Count"
    ).fillna(0).astype(int)
)

Market_Type,Other Market Names,Possible Uzhavar Sandhai
Report_Month,,
2025-10-01,33480,5237
2025-11-01,3061,432
2025-12-01,13556,3247
2026-01-01,15246,3887
2026-02-01,11600,3511
2026-03-01,8227,3349
2026-04-01,7667,2920



Active market-commodity counts:


Market_Type,Other Market Names,Possible Uzhavar Sandhai
Report_Month,,
2025-10-01,2533,350
2025-11-01,1679,247
2025-12-01,2118,354
2026-01-01,2244,369
2026-02-01,2171,424
2026-03-01,2006,422
2026-04-01,2074,804


In [30]:
# Examine conventional market series that reported after 6 November 2025,
# including series with fewer than 365 historical reporting days.

all_post_nov = series_metrics[
    (series_metrics["Last_Date"] > pd.Timestamp("2025-11-06"))
    & (~series_metrics["Market"].str.contains(
        r"uzhavar|sandhai|santhai",
        case=False,
        na=False
    ))
].copy()

post_nov_history_summary = (
    all_post_nov
    .groupby("Project_Commodity")
    .agg(
        Post_Nov_Series=("Reporting_Days", "size"),
        Median_Historical_Days=("Reporting_Days", "median"),
        P25_Historical_Days=("Reporting_Days", lambda x: x.quantile(0.25)),
        P75_Historical_Days=("Reporting_Days", lambda x: x.quantile(0.75)),
        Series_With_365_Days=(
            "Reporting_Days",
            lambda x: (x >= 365).sum()
        ),
        Latest_Report=("Last_Date", "max")
    )
    .reindex([
        "Cotton", "Grapes", "Onion",
        "Pomegranate", "Soybean", "Tomato"
    ])
    .reset_index()
)

display(post_nov_history_summary)

,Project_Commodity,Post_Nov_Series,Median_Historical_Days,P25_Historical_Days,P75_Historical_Days,Series_With_365_Days,Latest_Report
0,Cotton,457,6.0,2.0,18.0,0,2026-04-20
1,Grapes,301,6.0,2.0,13.0,0,2026-04-20
2,Onion,1427,7.0,2.0,22.0,0,2026-04-20
3,Pomegranate,293,7.0,2.0,18.0,0,2026-04-20
4,Soybean,836,6.0,2.0,18.0,0,2026-04-20
5,Tomato,1375,7.0,2.0,21.0,1,2026-04-20


In [31]:
# Measure historical depth of conventional series reporting after Nov 6, 2025

reporting_depth_summary = (
    all_post_nov
    .groupby("Project_Commodity")
    .agg(
        Total_Series=("Reporting_Days", "size"),

        At_30_Days=(
            "Reporting_Days",
            lambda x: (x >= 30).sum()
        ),

        At_60_Days=(
            "Reporting_Days",
            lambda x: (x >= 60).sum()
        ),

        At_90_Days=(
            "Reporting_Days",
            lambda x: (x >= 90).sum()
        ),

        At_180_Days=(
            "Reporting_Days",
            lambda x: (x >= 180).sum()
        ),

        At_365_Days=(
            "Reporting_Days",
            lambda x: (x >= 365).sum()
        )
    )
    .reindex([
        "Cotton", "Grapes", "Onion",
        "Pomegranate", "Soybean", "Tomato"
    ])
    .reset_index()
)

display(reporting_depth_summary)

,Project_Commodity,Total_Series,At_30_Days,At_60_Days,At_90_Days,At_180_Days,At_365_Days
0,Cotton,457,56,0,0,0,0
1,Grapes,301,12,0,0,0,0
2,Onion,1427,200,3,2,1,0
3,Pomegranate,293,35,0,0,0,0
4,Soybean,836,95,0,0,0,0
5,Tomato,1375,202,3,2,2,1


In [32]:
# Inventory every commodity in the original dataset

RAW_DIR = DATA_FILE.parents[1] / "raw" / "parquet"

commodity_inventory = selection_con.execute(f"""
    SELECT
        TRIM(Commodity) AS Commodity,

        COUNT(*) AS Total_Records,

        COUNT(DISTINCT State) AS States,

        COUNT(DISTINCT State || '|' || District || '|' || Market)
            AS Market_Identities,

        COUNT(DISTINCT TRY_CAST(Arrival_Date AS DATE))
            AS Reporting_Dates,

        MIN(TRY_CAST(Arrival_Date AS DATE)) AS First_Date,

        MAX(TRY_CAST(Arrival_Date AS DATE)) AS Last_Date

    FROM read_parquet(
        '{RAW_DIR.as_posix()}/*.parquet',
        union_by_name = true
    )

    WHERE Commodity IS NOT NULL

    GROUP BY TRIM(Commodity)

    ORDER BY Total_Records DESC
""").df()

print("Total distinct commodities:", len(commodity_inventory))

display(commodity_inventory)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total distinct commodities: 389


,Commodity,Total_Records,States,Market_Identities,Reporting_Dates,First_Date,Last_Date
0,Paddy (Dhan)(Common),3318434,31,2301,8918,2001-03-14,2025-11-06
1,Wheat,3260100,28,3275,9066,2001-01-10,2026-04-20
2,Potato,3248834,33,2990,9096,2001-01-10,2026-04-20
3,Onion,3168490,32,3223,9101,2001-01-10,2026-04-20
4,Tomato,2818577,32,3075,9072,2001-02-08,2026-04-20
...,...,...,...,...,...,...,...
384,spikenard,2,1,1,2,2025-05-03,2025-10-06
385,Egypian Clover (Barseem),2,1,2,2,2008-12-03,2011-08-25
386,Kuchur,2,2,2,2,2007-11-15,2011-11-01
387,sanay,2,1,1,2,2025-02-01,2025-02-27


In [33]:
# Identify vegetable commodities from the complete commodity inventory

vegetable_keywords = [
    "potato", "onion", "tomato", "brinjal", "eggplant",
    "cabbage", "cauliflower", "carrot", "radish", "beetroot",
    "cucumber", "bitter gourd", "bottle gourd", "ridge gourd",
    "snake gourd", "bitter gourd", "pumpkin", "ash gourd",
    "pointed gourd", "sponge gourd", "drumstick", "okra",
    "lady finger", "bhindi", "green chilli", "chilli",
    "capsicum", "bell pepper", "peas", "green peas",
    "beans", "french beans", "cluster beans", "cowpea",
    "spinach", "amaranthus", "methi", "fenugreek",
    "coriander leaves", "mint", "lettuce", "garlic",
    "ginger", "sweet potato", "yam", "colocasia",
    "tapioca", "mushroom", "broccoli", "turnip",
    "parwal", "pointed gourd", "ivy gourd", "tinda",
    "raw banana", "banana flower", "banana stem"
]

pattern = "|".join(
    keyword.replace(" ", r"\s+")
    for keyword in vegetable_keywords
)

vegetable_inventory = commodity_inventory[
    commodity_inventory["Commodity"].str.contains(
        pattern,
        case=False,
        na=False,
        regex=True
    )
].copy()

vegetable_inventory = vegetable_inventory.sort_values(
    "Total_Records",
    ascending=False
).reset_index(drop=True)

print("Potential vegetable commodities:", len(vegetable_inventory))

display(vegetable_inventory)

Potential vegetable commodities: 56


,Commodity,Total_Records,States,Market_Identities,Reporting_Dates,First_Date,Last_Date
0,Potato,3248834,33,2990,9096,2001-01-10,2026-04-20
1,Onion,3168490,32,3223,9101,2001-01-10,2026-04-20
2,Tomato,2818577,32,3075,9072,2001-02-08,2026-04-20
3,Brinjal,2448410,31,2897,9071,2001-01-27,2026-04-20
4,Green Chilli,2108000,31,2834,9019,2001-03-12,2026-04-20
5,Cauliflower,1701608,33,2397,9068,2001-01-27,2026-04-20
6,Bhindi (Ladies Finger),1572079,31,2493,9008,2001-04-08,2026-04-20
7,Cabbage,1505899,33,2572,8866,2001-11-22,2026-04-20
8,Bottle gourd,1215058,31,2083,8981,2001-04-12,2026-04-20
9,Bitter gourd,1181917,32,2389,8924,2001-04-21,2026-04-20


In [34]:
# Classify potential vegetable commodities

confirmed_vegetables = [
    "Potato", "Onion", "Tomato", "Brinjal", "Green Chilli",
    "Cauliflower", "Bhindi (Ladies Finger)", "Cabbage",
    "Bottle gourd", "Bitter gourd", "Pumpkin", "Carrot",
    "Ginger (Green)", "Garlic", "Capsicum", "Spinach",
    "Beetroot", "Peas Wet", "Drumstick", "Cowpea (Veg)",
    "Pointed gourd (Parval)", "Amaranthus",
    "French Beans (Frasbean)", "Beans", "Cluster beans",
    "Methi (Leaves)", "Elephant Yam (Suran)", "Tapioca",
    "Onion Green", "Tinda", "Chilly Capsicum",
    "Sweet Potato", "Green Peas", "Sweet Pumpkin",
    "Sponge gourd", "Mint (Pudina)", "Indian Beans (Seam)",
    "Yam (Ratalu)", "Cowpea (Lobia/Karamani)", "Turnip",
    "Surat Beans (Papadi)", "Bunch Beans", "White Pumpkin",
    "Yam", "Duster Beans",
    "Elephant Yam (Suran)/Amorphophallus",
    "Rat Tail Radish (Mogari)"
]

ambiguous_commodities = [
    "Peas cod",
    "Chilly Capsicum"
]

exclude_from_vegetable_scope = [
    "Black Gram (Urd Beans)(Whole)",
    "Ginger (Dry)",
    "Peas (Dry)",
    "Methi Seeds",
    "Guar Seed (Cluster Beans Seed)",
    "Dry Chillies",
    "Kabuli Chana (Chickpeas-White)",
    "White Peas"
]

vegetable_inventory["Classification"] = "Needs review"

vegetable_inventory.loc[
    vegetable_inventory["Commodity"].isin(confirmed_vegetables),
    "Classification"
] = "Vegetable"

vegetable_inventory.loc[
    vegetable_inventory["Commodity"].isin(ambiguous_commodities),
    "Classification"
] = "Ambiguous"

vegetable_inventory.loc[
    vegetable_inventory["Commodity"].isin(exclude_from_vegetable_scope),
    "Classification"
] = "Exclude"

print(vegetable_inventory["Classification"].value_counts())

display(
    vegetable_inventory[
        ["Commodity", "Classification", "Total_Records",
         "Market_Identities", "Last_Date"]
    ]
)

Classification
Vegetable    46
Exclude       8
Ambiguous     2
Name: count, dtype: int64


,Commodity,Classification,Total_Records,Market_Identities,Last_Date
0,Potato,Vegetable,3248834,2990,2026-04-20
1,Onion,Vegetable,3168490,3223,2026-04-20
2,Tomato,Vegetable,2818577,3075,2026-04-20
3,Brinjal,Vegetable,2448410,2897,2026-04-20
4,Green Chilli,Vegetable,2108000,2834,2026-04-20
5,Cauliflower,Vegetable,1701608,2397,2026-04-20
6,Bhindi (Ladies Finger),Vegetable,1572079,2493,2026-04-20
7,Cabbage,Vegetable,1505899,2572,2026-04-20
8,Bottle gourd,Vegetable,1215058,2083,2026-04-20
9,Bitter gourd,Vegetable,1181917,2389,2026-04-20


In [36]:
# Corrected vegetable reporting coverage screening

vegetable_screening_corrected = selection_con.execute(f"""
    WITH vegetable_prices AS (
        SELECT
            TRIM(Commodity) AS Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,
            TRY_CAST(Arrival_Date AS DATE) AS Report_Date,
            Modal_Price

        FROM read_parquet(
            '{RAW_DIR.as_posix()}/*.parquet',
            union_by_name = true
        )

        WHERE TRIM(Commodity) IN ({vegetable_sql_list})
          AND TRY_CAST(Arrival_Date AS DATE) IS NOT NULL
          AND Modal_Price IS NOT NULL
          AND ISFINITE(Modal_Price)
          AND Modal_Price > 0
    ),

    series AS (
        SELECT
            Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,

            COUNT(DISTINCT Report_Date) AS Total_Reporting_Days,

            COUNT(DISTINCT Report_Date) FILTER (
                WHERE Report_Date > DATE '2025-11-06'
            ) AS Post_Nov_Reporting_Days,

            MIN(Report_Date) AS First_Date,
            MAX(Report_Date) AS Last_Date

        FROM vegetable_prices

        GROUP BY
            Commodity, State, District, Market, Variety, Grade
    )

    SELECT
        Commodity,

        COUNT(*) AS Total_Series,

        COUNT(*) FILTER (
            WHERE Total_Reporting_Days >= 365
        ) AS Series_365_Historical_Days,

        COUNT(*) FILTER (
            WHERE Post_Nov_Reporting_Days >= 10
        ) AS Series_10_Post_Nov_Days,

        COUNT(*) FILTER (
            WHERE Post_Nov_Reporting_Days >= 30
        ) AS Series_30_Post_Nov_Days,

        COUNT(*) FILTER (
            WHERE Post_Nov_Reporting_Days >= 60
        ) AS Series_60_Post_Nov_Days,

        COUNT(*) FILTER (
            WHERE Post_Nov_Reporting_Days >= 90
        ) AS Series_90_Post_Nov_Days,

        COUNT(*) FILTER (
            WHERE Total_Reporting_Days >= 365
              AND Post_Nov_Reporting_Days >= 30
        ) AS Established_Series_30_Post_Nov_Days,

        MAX(Last_Date) AS Latest_Report

    FROM series

    GROUP BY Commodity
    ORDER BY Established_Series_30_Post_Nov_Days DESC,
             Series_365_Historical_Days DESC
""").df()

display(vegetable_screening_corrected)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Commodity,Total_Series,Series_365_Historical_Days,Series_10_Post_Nov_Days,Series_30_Post_Nov_Days,Series_60_Post_Nov_Days,Series_90_Post_Nov_Days,Established_Series_30_Post_Nov_Days,Latest_Report
0,Potato,7195,1806,812,253,0,0,1,2026-04-20
1,Onion,7488,1794,803,351,2,0,0,2026-04-20
2,Tomato,6458,1609,782,281,0,0,0,2026-04-20
3,Brinjal,4869,1389,729,259,0,0,0,2026-04-20
4,Green Chilli,3917,1190,649,299,0,0,0,2026-04-20
5,Bhindi (Ladies Finger),3297,1043,456,276,3,0,0,2026-04-20
6,Cauliflower,3649,1031,619,266,0,0,0,2026-04-20
7,Cabbage,3430,938,590,271,0,0,0,2026-04-20
8,Bitter gourd,3190,819,429,165,0,0,0,2026-04-20
9,Pumpkin,2698,804,434,226,3,0,0,2026-04-20


## Final Findings and Forecasting Scope

The forecasting suitability analysis expanded the initial six-commodity scope to 46 identified vegetable commodities using the original annual Parquet files.

The analysis evaluated historical reporting coverage, the number of market-variety-grade time series, reporting recency and continuity.

### Key Findings

1. Major vegetables, including Potato, Onion, Tomato, Brinjal, Green Chilli, Bhindi and Cauliflower, have substantial historical market data.
2. Thousands of individual market-variety-grade series are available for historical analysis.
3. A major reporting decline occurred around November 2025, affecting multiple commodities and market types.
4. Although reporting resumed for many series, very few combine extensive historical coverage with sufficient post-November observations.
5. The dataset's latest available date is April 20, 2026, limiting its suitability for live market-price predictions.

### Project Decisions

- Expand historical market intelligence to the identified vegetable commodities.
- Retain commodity, state, district, market, variety and grade as important dimensions.
- Do not assume that every market series is suitable for forecasting.
- Conduct historical forecasting experiments only on series that pass continuity and data-quality checks.
- Seek a more recent and reliable data source before presenting operational forecasts.

### Conclusion

The dataset is suitable for developing a broad historical agricultural market intelligence platform. Forecasting remains a selective experimental component until adequate reporting continuity and data freshness can be established.